In [ ]:
# Colab setup: Python 3.13 compatible SentencePiece. Run this cell first.
import importlib.util, subprocess, sys
print("Python:", sys.version.split()[0])
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--prefer-binary", "transformers==4.49.0", "sentencepiece==0.2.1"])
print("Dependencies installed.")


In [ ]:
import json, time
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
torch.set_num_threads(2)
MODEL_ID = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, trust_remote_code=False, use_safetensors=True).to('cpu').eval()
context = 'You can check your application status using its reference number.'
settings = {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 48}


In [ ]:
questions = [
 'How can I check my application status?',
 'What do I need to check my application status?',
 'How can I find out the status of my application?']
rows=[]
for i, question in enumerate(questions,1):
    prompt=f'Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:'
    inputs=tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    started=time.perf_counter()
    with torch.inference_mode():
        output=model.generate(**inputs, **settings)
    seconds=time.perf_counter()-started
    answer=tokenizer.decode(output[0],skip_special_tokens=True).strip()
    assert answer, f'Empty answer for E{i:02d}'
    keyword_hit = 'reference' in answer.lower() and 'number' in answer.lower()
    semantic_ok = ('reference number' in answer.lower() and
                   not any(term in answer.lower() for term in ('fee','cost','tomorrow','approved','rejected')))
    rows.append({'id':f'E{i:02d}','question':question,'answer':answer,
                 'reference_keyword_hit':keyword_hit,'manual_context_review': 'Supported: directs the user to the reference number without introducing unsupported application information.' if semantic_ok else 'Review manually: answer may be incomplete or unsupported',
                 'automatic_semantic_heuristic':
                 'Supported by synthetic context' if semantic_ok else 'Needs manual inspection: possibly incomplete or unsupported',
                 'input_tokens':int(inputs['input_ids'].shape[-1]),
                 'output_tokens_including_special':int(output.shape[-1]),
                 'inference_seconds':round(seconds,3)})
report={'mode':'real_local_model','model':MODEL_ID,        'device':'cpu','context':context,'generation':settings,
        'metric':'reference+number keyword indicator, not semantic accuracy',
        'keyword_hits':sum(r['reference_keyword_hit'] for r in rows),'total':len(rows),
        'external_inference_api_calls':0,'billed_api_cost_usd':0,
        'cases':rows,'limitation':'Three synthetic English examples cannot measure general accuracy or security.',
        'improvement':'Add diverse questions, answer-grounding checks, and negative cases.',
        'manual_review_note':'Review judgment based on supplied synthetic context; keyword metric is separate from meaning.'}
Path('murshid_live_outputs').mkdir(exist_ok=True)
Path('murshid_live_outputs/evaluation_result.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(json.dumps(report,indent=2))
